# 02. FT-Transformer v2 + Inverse-Prior Correction

Kaggle **Predicting Student Health Risk (Playground Series S6E7)** 모델링 노트북입니다.

EDA에서 확인한 심한 클래스 불균형을 고려해 FT-Transformer와 inverse-prior correction을 결합합니다. 현재 제출 기준 Private Balanced Accuracy는 **0.95066**으로, 본 프로젝트에서 가장 높은 단일 모델 점수입니다.

### Pipeline
1. 원본 13개 feature
2. 5-fold cross-fitted multiclass exact-value Target Encoding: 13 × 3 = 39개 추가 feature
3. 총 52개 input으로 FT-Transformer 학습
4. 7-fold Stratified CV, fold별 4-member ensemble
5. test probability 평균
6. inverse-prior correction 후 최종 class 결정

> 핵심 아이디어: 데이터의 약 86%가 `at-risk`이므로 raw argmax는 다수 클래스 쪽으로 기울 수 있습니다. Balanced Accuracy는 각 클래스 recall을 동일하게 평가하므로, 각 클래스 probability를 class prior로 나누는 inverse-prior correction으로 decision boundary를 보정합니다.

In [ ]:
%pip install -q catstat==0.4.0 masamlp==0.4.0


In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

TARGET = 'health_condition'
ID_COL = 'id'
DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e7')
OUTPUT_DIR = Path('/kaggle/working/ftt_v2_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 7
N_ENS = 4
N_EPOCHS = 16
SEED = 42


## 1. Load data
EDA 단계에서 사용했던 동일한 train/test 구조를 그대로 사용합니다.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

feature_columns = [c for c in train.columns if c not in [ID_COL, TARGET]]
categorical_columns = [c for c in feature_columns if train[c].dtype == 'object']

le = LabelEncoder()
y = le.fit_transform(train[TARGET])
raw_train = train[feature_columns].copy()
raw_test = test[feature_columns].copy()

prior = np.bincount(y, minlength=3).astype(float) / len(y)
print('features:', len(feature_columns))
print('classes:', le.classes_)
print('class prior:', prior)


## 2. Exact-value multiclass Target Encoding
각 원본 feature에 대해 세 클래스별 Target Encoding을 생성합니다. 13개 feature × 3 classes = 39개 TE feature이며, 원본 13개와 합쳐 총 52개 input을 사용합니다.

Outer validation fold의 정답이 encoding에 들어가는 leakage를 막기 위해 각 outer-fold의 training data 내부에서 다시 5-fold cross-fitting을 수행합니다.

In [ ]:
def add_exact_value_te(fit_raw, valid_raw, test_raw, y_fit, feature_columns):
    from catstat import TargetEncoder

    encoder = TargetEncoder(
        cols=feature_columns,
        stats=('mean',),
        target_type='multiclass',
        smooth='auto',
        numeric='direct',
        cv=5,
        random_state=SEED,
        output='numpy',
    )

    fit_te = np.asarray(encoder.fit_transform(fit_raw[feature_columns], y_fit), dtype=np.float32)
    valid_te = np.asarray(encoder.transform(valid_raw[feature_columns]), dtype=np.float32)
    test_te = np.asarray(encoder.transform(test_raw[feature_columns]), dtype=np.float32)

    names = [f'exact_te_{j:02d}' for j in range(len(feature_columns) * 3)]

    def attach(raw, encoded):
        raw = raw[feature_columns].reset_index(drop=True).copy()
        te = pd.DataFrame(encoded, columns=names, index=raw.index)
        return pd.concat([raw, te], axis=1)

    return attach(fit_raw, fit_te), attach(valid_raw, valid_te), attach(test_raw, test_te)


## 3. FT-Transformer
사용한 주요 설정은 `d_block=128`, `n_blocks=2`, `8 attention heads`, 16 epochs입니다. Fold마다 4개의 독립 ensemble member를 평균해 neural-network variance를 줄입니다.

In [ ]:
def make_model(categorical_columns):
    from masamlp import MasaClassifier

    return MasaClassifier(
        model='ft_transformer',
        model_params={
            'd_block': 128,
            'n_blocks': 2,
            'attention_n_heads': 8,
        },
        n_epochs=N_EPOCHS,
        batch_size=4096,
        eval_batch_size=8192,
        learning_rate=1e-3,
        weight_decay=1e-5,
        optimizer='adamw',
        lr_scheduler='cosine',
        num_embedding='plr-lite',
        numeric_scaler='quantile',
        categorical_features=categorical_columns,
        cat_encoding='embedding',
        n_ens=N_ENS,
        ens_mode='loop',
        early_stopping_rounds=None,
        class_weight=None,
        device='cuda',
        amp='auto',
        verbose=1,
        random_state=SEED,
    )

def normalize_probability(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)


## 4. 7-fold Stratified CV
각 fold에서 Target Encoding을 fold-local하게 생성한 뒤 FT-Transformer를 학습합니다. Test probability는 7개 fold 결과를 평균합니다.

In [ ]:
oof = np.zeros((len(train), 3), dtype=np.float32)
test_sum = np.zeros((len(test), 3), dtype=np.float64)

outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (fit_idx, valid_idx) in enumerate(outer.split(raw_train, y), start=1):
    print(f'\n===== Fold {fold}/{N_SPLITS} =====')

    fit_raw = raw_train.iloc[fit_idx].reset_index(drop=True)
    valid_raw = raw_train.iloc[valid_idx].reset_index(drop=True)
    test_raw = raw_test.reset_index(drop=True)

    fit_frame, valid_frame, test_frame = add_exact_value_te(
        fit_raw, valid_raw, test_raw, y[fit_idx], feature_columns
    )
    assert fit_frame.shape[1] == 52

    model = make_model(categorical_columns)
    model.fit(fit_frame, y[fit_idx])

    valid_proba = normalize_probability(model.predict_proba(valid_frame)).astype(np.float32)
    test_proba_fold = normalize_probability(model.predict_proba(test_frame)).astype(np.float32)

    oof[valid_idx] = valid_proba
    test_sum += test_proba_fold

    del model, fit_frame, valid_frame, test_frame
    gc.collect()

test_proba = normalize_probability(test_sum / N_SPLITS).astype(np.float32)


## 5. Inverse-prior correction
Raw prediction은 가장 큰 probability를 그대로 선택합니다. 하지만 `at-risk`가 압도적으로 많은 데이터에서는 다수 클래스 쏠림이 생길 수 있습니다.

Inverse prior는 다음처럼 각 클래스 probability를 그 클래스의 prior로 나눈 뒤 argmax를 취합니다.

`adjusted_score(c) = predicted_probability(c) / class_prior(c)`

즉, 원래 흔한 클래스는 상대적으로 억제하고 드문 클래스는 상대적으로 강화합니다. 이는 각 클래스 recall을 동일하게 평가하는 Balanced Accuracy와 잘 맞습니다.

In [ ]:
raw_pred = np.argmax(test_proba, axis=1)
inverse_prior_pred = np.argmax(test_proba / prior, axis=1)

submission = sample.copy()
submission[TARGET] = le.inverse_transform(inverse_prior_pred)
submission.to_csv(OUTPUT_DIR / 'submission_ftt_v2_inverse_prior.csv', index=False)
submission.head()


## Result
최종 Kaggle 제출에서는 **inverse-prior variant**가 가장 높은 점수를 기록했습니다.

- Private Balanced Accuracy: **0.95066**
- Public Balanced Accuracy: **0.95054**

이 결과를 단순히 ‘Transformer가 LightGBM보다 좋다’고 해석하기보다는, **(1) class-specific Target Encoding, (2) FT-Transformer의 feature interaction 학습, (3) fold별 ensemble, (4) Balanced Accuracy에 맞춘 inverse-prior decision correction**이 함께 작동한 결과로 보는 것이 적절합니다.

### Project flow
`EDA → class imbalance 확인 → feature/target 관계 분석 → FT-Transformer + Target Encoding → inverse-prior correction → Kaggle submission`
